# 🤖 AI Engineering Fundamentals — Lezione 5
## Notebook Gruppo B

**ITS Novitas 4.0 | Giovedì 04/06/2026**

---

### 📋 Istruzioni
1. **File → Salva una copia in Drive** prima di iniziare
2. Lavorate in gruppo — discutete prima di scrivere
3. Alla fine: **File → Scarica → .ipynb** e caricate su GitHub

### 👥 Membri del gruppo

In [1]:
GRUPPO = "B"
MEMBRI = ["Anna", "Lorenzo", "Stefano", ""]  # ← inserite i vostri nomi
print(f"Gruppo {GRUPPO} — {', '.join(m for m in MEMBRI if m)}")

Gruppo B — Anna, Lorenzo, Stefano


In [2]:
!pip install anthropic requests -q
!pip install anthropic chromadb

from google.colab import userdata
import anthropic, os, requests, chromadb

os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
client = anthropic.Anthropic()
chroma_client = chromadb.Client() #

# Tool loop base — usatelo per tutti gli esercizi
def tool_loop(messaggio, tools, esegui_fn, system=None):
    """Loop agente generico con contatore di sicurezza."""
    history = [{"role": "user", "content": messaggio}]
    iterazioni = 0
    while True:
        iterazioni += 1
        if iterazioni > 10:
            return "Errore: loop non terminato"
        params = {"model": "claude-haiku-4-5-20251001",
                  "max_tokens": 1024, "tools": tools, "messages": history}
        if system:
            params["system"] = system
        response = client.messages.create(**params)
        if response.stop_reason == "end_turn":
            return next(b.text for b in response.content if b.type == "text")
        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})
            results = []
            for b in response.content:
                if b.type == "tool_use":
                    print(f"  🔧 {b.name}({b.input})")
                    r = esegui_fn(b.name, b.input)
                    print(f"  ✅ {str(r)[:100]}")
                    results.append({"type": "tool_result",
                                   "tool_use_id": b.id, "content": str(r)})
            history.append({"role": "user", "content": results})

print("✅ Setup completato!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 838.7/838.7 kB 11.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 114.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.9 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    

---
## 🎯 Tema del Gruppo B: Costruire Tool Personalizzati

Costruite 3 tool reali per il chatbot WiData:
calcolatrice sicura, meteo in tempo reale, Wikipedia.
Poi integrateli in un chatbot multi-tool.

---
### Esercizio 1 — Tool calcolatrice con validazione *(guidato)*

Costruite una calcolatrice sicura che usa `eval()` ma valida
l'input prima di eseguirlo. Dimostrate che blocca input malevoli.

In [3]:
# Esercizio 1 — calcolatrice sicura

tool_calcolatrice = {
    "name": "calcola",
    "description": "Esegui un'operazione matematica precisa. Usa per calcoli aritmetici, percentuali, potenze.",
    "input_schema": {
        "type": "object",
        "properties": {
            "espressione": {
                "type": "string",
                "description": "Espressione matematica. Es: '234 * 567', '15/100 * 847'"
            }
        },
        "required": ["espressione"]
    }
}

def calcola(espressione: str) -> str:
    # TODO: implementate la validazione con lista bianca
    # Caratteri consentiti: cifre, operatori, parentesi, spazi, punto
    allowed = set('0123456789+-*/().% ')
    if not all(c in allowed for c in espressione):
        return "caratteri non validi"  # messaggio di errore
    try:
        risultato = eval(espressione)
        return f"{espressione} = {risultato}"
    except Exception as e:
        return f"Errore nel calcolo: {str(e)}"

# Test sicurezza
test_input = [
    "234 * 567",                    # ✅ valido
    "15 / 100 * 847",               # ✅ valido
    "__import__('os').system('ls')", # ❌ da bloccare
    "open('/etc/passwd').read()",    # ❌ da bloccare
]

print("Test sicurezza calcolatrice:")
for inp in test_input:
    risultato = calcola(inp)
    stato = "✅" if "Errore" in risultato or "non valida" in risultato or "=" in risultato else "⚠️"
    print(f"{stato} Input: '{inp[:40]}' → {risultato}")

# Test con il chatbot
def esegui(nome, params):
    if nome == "calcola": return calcola(params["espressione"])
    return "Tool non trovato"

print()
print(tool_loop("Quanto fa il 22% di 1500?", [tool_calcolatrice], esegui))

Test sicurezza calcolatrice:
✅ Input: '234 * 567' → 234 * 567 = 132678
✅ Input: '15 / 100 * 847' → 15 / 100 * 847 = 127.05
⚠️ Input: '__import__('os').system('ls')' → caratteri non validi
⚠️ Input: 'open('/etc/passwd').read()' → caratteri non validi

  🔧 calcola({'espressione': '22/100 * 1500'})
  ✅ 22/100 * 1500 = 330.0
Il 22% di 1500 è **330**.


---
### Esercizio 2 — Tool meteo con API reale *(guidato)*

Implementate il tool meteo usando open-meteo.com.
Gratuito, senza API key, funziona subito.

In [4]:
# Esercizio 2 — tool meteo

CITTA_COORD = {
    "sassari":  {"lat": 40.7259, "lon": 8.5563},
    "cagliari": {"lat": 39.2238, "lon": 9.1217},
    "nuoro":    {"lat": 40.3207, "lon": 9.3311},
    "olbia":    {"lat": 40.9237, "lon": 9.4992},
    "roma":     {"lat": 41.9028, "lon": 12.4964},
    "milano":   {"lat": 45.4642, "lon": 9.1900},
}

tool_meteo = {
    "name": "get_meteo",
    "description": "Ottieni il meteo attuale per una città italiana. Usa quando l'utente chiede del tempo atmosferico.",
    "input_schema": {
        "type": "object",
        "properties": {
            "citta": {
                "type": "string",
                "description": "Nome città in minuscolo. Es: 'sassari', 'cagliari', 'roma'"
            }
        },
        "required": ["citta"]
    }
}

def get_meteo(citta: str) -> str:
    citta = citta.lower().strip()
    if citta not in CITTA_COORD:
        return f"Città '{citta}' non supportata. Disponibili: {', '.join(CITTA_COORD.keys())}"

    coord = CITTA_COORD[citta]
    # TODO: chiamate l'API open-meteo.com
    # URL: https://api.open-meteo.com/v1/forecast
    # Parametri: latitude, longitude, current=temperature_2m,weathercode,windspeed_10m
    # timezone=Europe/Rome
    url = (
        f"https://api.open-meteo.com/v1/forecast"
        f"?latitude={coord['lat']}&longitude={coord['lon']}"
        f"&current=temperature_2m,windspeed_10m,relative_humidity_2m"
        f"&timezone=Europe/Rome"
    )
    try:
        # TODO: fate la chiamata con timeout=5
        data = requests.get(url, timeout=5).json()
        curr = data["current"]
        return (
            f"Meteo a {citta.title()}: {curr['temperature_2m']}°C, "
            f"umidità {curr['relative_humidity_2m']}%, "
            f"vento {curr['windspeed_10m']} km/h."
        )
    except Exception as e:
        return f"Errore API meteo: {str(e)}"

# Test diretto
print(get_meteo("sassari"))
print(get_meteo("parigi"))  # città non in lista

# Test con chatbot
def esegui_v2(nome, params):
    if nome == "calcola":  return calcola(params["espressione"])
    if nome == "get_meteo": return get_meteo(params["citta"])
    return "Tool non trovato"

print()
print(tool_loop("Che tempo fa a Sassari e Cagliari?",
                [tool_calcolatrice, tool_meteo], esegui_v2))

Meteo a Sassari: 27.1°C, umidità 51%, vento 6.9 km/h.
Città 'parigi' non supportata. Disponibili: sassari, cagliari, nuoro, olbia, roma, milano

  🔧 get_meteo({'citta': 'sassari'})
  ✅ Meteo a Sassari: 27.1°C, umidità 51%, vento 6.9 km/h.
  🔧 get_meteo({'citta': 'cagliari'})
  ✅ Meteo a Cagliari: 26.6°C, umidità 52%, vento 13.2 km/h.
Ecco il meteo attuale:

**Sassari:**
- Temperatura: 27.1°C
- Umidità: 51%
- Vento: 6.9 km/h

**Cagliari:**
- Temperatura: 26.6°C
- Umidità: 52%
- Vento: 13.2 km/h

In entrambe le città il tempo è bello, con temperature simili intorno ai 27°C. Cagliari ha un vento leggermente più forte rispetto a Sassari.


---
### Esercizio 3 — Tool Wikipedia *(libero)*

Costruite il tool Wikipedia con fallback search.
Se la pagina non esiste, cercate con l'API di ricerca
e restituite il primo risultato.

In [5]:
# Esercizio 3 — tool Wikipedia con fallback

tool_wikipedia = {
    "name": "cerca_wikipedia",
    "description": "Cerca informazioni su Wikipedia. Usa per fatti, biografie, concetti tecnici, eventi storici.",
    "input_schema": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Termine da cercare su Wikipedia (in italiano o inglese)"
            },
            "lingua": {
                "type": "string",
                "description": "Lingua Wikipedia: 'it' per italiano, 'en' per inglese",
                "enum": ["it", "en"]
            }
        },
        "required": ["query"]
    }
}

HEADERS = {
    "User-Agent": "WiDataChatbot/1.0 (corso ITS Novitas; gruppoB@widata.cloud)"
}

def cerca_wikipedia(query: str, lingua: str = "it") -> str:
    try:
        # Tentativo diretto
        url = f"https://{lingua}.wikipedia.org/api/rest_v1/page/summary/{query.replace(' ', '_')}"
        r = requests.get(url, timeout=5, headers=HEADERS)

        if r.status_code == 404:
            # TODO: fallback — usate l'API di ricerca Wikipedia
            # https://{lingua}.wikipedia.org/w/api.php
            # action=opensearch, search=query, limit=1

            search = requests.get(
                f"https://{lingua}.wikipedia.org/w/api.php",
                params={"action": "opensearch", "search": query,
                        "format": "json", "limit": 1},
                timeout=5,
                headers=HEADERS
            )
            if search.status_code != 200 or not search.text.strip():
                return f"Nessun risultato per '{query}'"

            dati_search = search.json()
            if not dati_search[1]:
                return f"Nessun risultato per '{query}' su Wikipedia {lingua}"

            titolo = dati_search[1][0]
            r = requests.get(
                f"https://{lingua}.wikipedia.org/api/rest_v1/page/summary/{titolo.replace(' ', '_')}",
                timeout=5,
                headers=HEADERS
            )

        if r.status_code != 200:
            return f"Wikipedia non disponibile (status {r.status_code})"
        if not r.text.strip():
            return f"Wikipedia ha risposto vuoto per '{query}'"

        data = r.json()
        extract = data.get("extract", "Nessuna descrizione disponibile")
        if len(extract) > 50:
            extract = extract[:50] + "..."
        return f"Wikipedia ({data.get('title', query)}): {extract}"

    except requests.Timeout:
        return "Errore: Wikipedia non risponde (timeout)"
    except Exception as e:
        return f"Errore Wikipedia: {str(e)}"

        data = r.json()
        extract = data.get("extract", "Nessuna descrizione")
        return f"{data.get('title', query)}: {extract[:500]}"

    except requests.Timeout:
        return "Errore: Wikipedia non risponde (timeout)"
    except Exception as e:
        return f"Errore Wikipedia: {str(e)}"

# Test
print(cerca_wikipedia("Sassari"))
print(cerca_wikipedia("Anthropic", "en"))
print(cerca_wikipedia("termine che non esiste xyzxyz"))  # test fallback

# Aggiornate il router e testate con il chatbot
def esegui_v3(nome, params):
    if nome == "calcola":          return calcola(params["espressione"])
    if nome == "get_meteo":        return get_meteo(params["citta"])
    if nome == "cerca_wikipedia":  return cerca_wikipedia(params["query"], params.get("lingua", "it"))
    return "Tool non trovato"

TUTTI_I_TOOL = [tool_calcolatrice, tool_meteo, tool_wikipedia]

print()
print(tool_loop("Chi ha fondato Anthropic e in che anno?",
                TUTTI_I_TOOL, esegui_v3))

Wikipedia (Sassari): Sassari è un comune italiano di 120 185 abitanti, ...
Wikipedia (Anthropic): Anthropic PBC is an American artificial intelligen...
Nessun risultato per 'termine che non esiste xyzxyz' su Wikipedia it

  🔧 cerca_wikipedia({'query': 'Anthropic', 'lingua': 'it'})
  ✅ Wikipedia (Anthropic): Anthropic è una benefit corporation statunitense d...
Secondo le informazioni di Wikipedia, **Anthropic** è stata fondata nel **2021**.

La società è stata fondata da un gruppo di ricercatori, tra cui **Dario Amodei** e **Daniela Amodei**, che in precedenza lavoravano presso OpenAI. Anthropic è un'azienda americana specializzata nello sviluppo di intelligenza artificiale, ed è nota soprattutto per aver creato Claude, un assistente AI conversazionale come quello con cui stai parlando.


---
### Esercizio 4 — Chatbot completo con 3 tool *(libero)*

Integrate i 3 tool nel chatbot RAG della Lezione 4.
Il chatbot deve usare sia il contesto RAG che i tool
quando appropriato.

In [6]:
# Esercizio 4 — chatbot completo RAG + 3 tool

# TODO: integrate chromadb e il documento WiData dalla Lezione 4 V
# Il chatbot deve:
# 1. Recuperare chunk RAG rilevanti per la domanda
# 2. Passarli come contesto nel system prompt
# 3. Avere a disposizione i 3 tool
# 4. Rispondere usando RAG per domande sui prodotti
#    e tool per calcoli, meteo, Wikipedia

DOCUMENTO_WIDATA = """
WiData Srl — Manuale Prodotti IoT

SENSORE XS200 - MONITORAGGIO AMBIENTALE
Il sensore XS200 è progettato per il monitoraggio ambientale in ambienti industriali e urbani.
Misura temperatura (-20°C a +60°C), umidità relativa (0-100%), pressione atmosferica e qualità dell'aria (CO2, PM2.5).
Classificazione IP67: impermeabile e resistente alla polvere. Alimentazione: batteria Li-Ion 3.7V, autonomia 2 anni.
Connettività: LoRaWAN, NB-IoT, WiFi 802.11n. Dimensioni: 85x45x30mm. Peso: 120g.
Certificazioni: CE, FCC, RoHS. Garanzia: 3 anni.

GATEWAY GW500 - CONCENTRATORE DATI
Il gateway GW500 raccoglie dati da fino a 1000 sensori simultaneamente tramite LoRaWAN.
Copertura fino a 15km in aree rurali, 3km in aree urbane. Elaborazione edge computing integrata.
Connessione cloud via Ethernet, WiFi o 4G LTE. Storage locale: 32GB SSD.
Alimentazione: 220V AC o pannello solare (opzionale). Temperatura operativa: -40°C a +70°C.
Certificazioni: CE, IP65. Installazione: palo, tetto o rack.

PIATTAFORMA XPLORE - ANALYTICS
Xplore è la piattaforma cloud di WiData per la visualizzazione e analisi dei dati IoT.
Dashboard personalizzabili con grafici real-time, storico dati fino a 5 anni.
Alerting automatico via email, SMS o webhook quando i valori superano soglie configurabili.
API REST per integrazione con sistemi terzi (ERP, SCADA, BIM).
Machine learning per previsione anomalie e manutenzione predittiva.
Piani: Free (5 sensori), Pro (100 sensori, €49/mese), Enterprise (illimitato, prezzo su richiesta).
SLA: 99.9% uptime garantito nei piani Pro ed Enterprise.

SUPPORTO E ASSISTENZA
Supporto tecnico disponibile lunedì-venerdì 9:00-18:00.
Email: support@widata.cloud | Telefono: +39 079 123456.
Per informazioni commerciali: sales@widata.cloud.
Sede: Via Roma 42, Sassari (SS) 07100, Italia.
Spedizioni in tutta Italia in 3-5 giorni lavorativi.
"""





SYSTEM_COMPLETO = """
Sei l'assistente di WiData Srl.
Per domande sui prodotti WiData usa il contesto fornito.
Per calcoli usa la calcolatrice.
Per il meteo usa il tool meteo.
Per informazioni generali usa Wikipedia.
Non inventare mai dati sui prodotti WiData.
"""

# Test con domande miste
domande_miste = [
    "Quanto costa il piano Pro di Xplore per 3 anni?",     # RAG + calcolatrice
    "Fa più caldo a Sassari o Cagliari oggi?",              # meteo 2 città
    "Chi ha inventato LoRaWAN, la tecnologia usata da XS200?",  # Wikipedia + RAG
]

def chunka_testo(testo, chunk_size=400, overlap=50):
    """Divide il testo in chunk con overlap."""
    chunks = []
    start = 0
    while start < len(testo):
        end = start + chunk_size
        chunk = testo[start:end]
        if chunk.strip():  # ignora chunk vuoti
            chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

def esegui_tool_completo(nome, parametri):
    """Router che gestisce sia tool custom che tool MCP."""
    # Tool custom
    if nome == "calcola":           return calcola(parametri["espressione"])
    if nome == "get_meteo":         return get_meteo(parametri["citta"])
    if nome == "cerca_wikipedia":   return cerca_wikipedia(parametri["query"], parametri.get("lingua", "it"))


def chatbot_completo(messaggio, history, collection):
    """Chatbot con RAG + tool + storia + streaming."""

    # 1. Recupera chunk RAG rilevanti
    chunks = []
    if collection is not None:
        risultati = collection.query(query_texts=[messaggio], n_results=3)
        chunks = risultati["documents"][0]

    # 2. Costruisci il messaggio con contesto RAG
    if chunks:
        contesto = "\n\n---\n\n".join(chunks)
        messaggio_con_rag = f"""Documenti di riferimento:

{contesto}

---

Domanda: {messaggio}"""
    else:
        messaggio_con_rag = messaggio

    # 3. Aggiungi alla history
    history.append({"role": "user", "content": messaggio_con_rag})

    # 4 + 5. Tool loop
    iterazioni = 0
    while True:
        iterazioni += 1
        if iterazioni > 10:
            print("⚠️ Loop non terminato")
            break

        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=800,
            system=SYSTEM_COMPLETO,
            tools=TUTTI_I_TOOL,
            messages=history
        )

        # Tool richiesto — esegui e rimanda
        if response.stop_reason == "tool_use":
            history.append({"role": "assistant", "content": response.content})
            tool_results = []
            for block in response.content:
                if block.type == "tool_use":
                    print(f"  🔧 {block.name}({block.input})")
                    risultato = esegui_tool_completo(block.name, block.input)
                    print(f"  ✅ {str(risultato)[:100]}")
                    tool_results.append({
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": str(risultato)
                    })
            history.append({"role": "user", "content": tool_results})
            continue

        # 6. Risposta finale — streaming
        if response.stop_reason == "end_turn":
            history.append({"role": "assistant", "content": response.content})
            testo_completo = ""

            # Richiama con streaming per la risposta finale
            print("\n🤖 ", end="", flush=True)
            with client.messages.stream(
                model="claude-haiku-4-5-20251001",
                max_tokens=800,
                system=SYSTEM_COMPLETO,
                messages=history[:-1]  # escludi l'ultima risposta già generata
            ) as stream:
                for text in stream.text_stream:
                    testo_completo += text
                    print(text, end="", flush=True)
            print("\n")

            # Aggiorna l'ultimo messaggio in history con il testo completo
            history[-1] = {"role": "assistant", "content": testo_completo}
            return testo_completo

def main():
    # Setup ChromaDB
    import chromadb
    chroma_client = chromadb.Client()
    collection = chroma_client.get_or_create_collection("widata_main")

    # Indicizza il documento WiData se vuoto
    if collection.count() == 0:
        chunks = chunka_testo(DOCUMENTO_WIDATA)
        collection.add(documents=chunks, ids=[str(i) for i in range(len(chunks))])
        print(f"✅ {collection.count()} chunk indicizzati\n")

    history = []




    for domanda in domande_miste:

        domanda = domanda.strip()
        print(domanda)
        chatbot_completo(domanda, history, collection)

main()

# ...

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 66.4MiB/s]


✅ 6 chunk indicizzati

Quanto costa il piano Pro di Xplore per 3 anni?
  🔧 calcola({'espressione': '49 * 12 * 3'})
  ✅ 49 * 12 * 3 = 1764

🤖 Il piano Pro di Xplore costa **€49 al mese**.

Per un periodo di 3 anni, il costo totale è:
- **€1.764** (49 € × 12 mesi × 3 anni)

Ricorda che il piano Pro include fino a 100 sensori e garantisce un SLA del 99.9% di uptime.

Se hai esigenze diverse o desideri informazioni commerciali personalizzate, puoi contattare il team di WiData a: **sales@widata.cloud**

Fa più caldo a Sassari o Cagliari oggi?
  🔧 get_meteo({'citta': 'sassari'})
  ✅ Meteo a Sassari: 27.1°C, umidità 51%, vento 6.9 km/h.
  🔧 get_meteo({'citta': 'cagliari'})
  ✅ Meteo a Cagliari: 26.6°C, umidità 52%, vento 13.2 km/h.

🤖 Secondo i dati meteorologici di oggi:

- **Sassari**: 27,1°C (umidità 51%, vento 6,9 km/h)
- **Cagliari**: 26,6°C (umidità 52%, vento 13,2 km/h)

**Fa più caldo a Sassari** con una differenza di 0,5°C. 

Interessante notare che Sassari, dove ha sede WiData, ha a

---
## 📊 Preparate la presentazione (5 slide)

1. **La struttura di un tool** — i 3 campi con il vostro esempio migliore
2. **Validazione dell'input** — il test di sicurezza della calcolatrice
3. **Tool meteo in azione** — demo live con temperatura reale
4. **Tool concatenati** — domanda che usa RAG + calcolatrice insieme
5. **La vostra guida pratica** — quando usare RAG vs tool per rispondere

---
*ITS Novitas 4.0 — AI Engineering Fundamentals | Marco Uras*